In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
import matplotlib.pyplot as plt

In [3]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


#### resident populuation and above 60 population per planning area and subzone

In [3]:
population_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/visualisation_data/pln_area_with_population_data_2010.gpkg")
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/visualisation_data/pln_area_with_population_data_2015.gpkg")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/visualisation_data/pln_area_with_population_data_2020.gpkg")

population_pln_area_2010  = gpd.read_file(population_2010_filepath)
population_pln_area_2015  = gpd.read_file(population_2015_filepath)
population_pln_area_2015 = population_pln_area_2015.rename(columns = {"total_total": "total"})
population_pln_area_2020  = gpd.read_file(population_2020_filepath)

In [4]:
pln_area_2010 = population_pln_area_2010["planning_area"].tolist()
print(f"Number of planning areas in 2010 data: {len(pln_area_2010)}")
pln_area_2015 = population_pln_area_2015["planning_area"].tolist()
print(f"Number of planning areas in 2015 data: {len(pln_area_2015)}")
pln_area_2020 = population_pln_area_2020["planning_area"].tolist()
print(f"Number of planning areas in 2020 data: {len(pln_area_2020)}")

if pln_area_2015.sort() == pln_area_2020.sort():
    print("The planning areas for resident population data in 2015 and 2020 are the same")
else:
    print("NOT THE SAME")


Number of planning areas in 2010 data: 35
Number of planning areas in 2015 data: 55
Number of planning areas in 2020 data: 55
The planning areas for resident population data in 2015 and 2020 are the same


In [5]:
# check what planning areas are different between 2015 and 2010
set_2010 = set(pln_area_2010)
set_2015 = set(pln_area_2015)

diff_in_2010 = set_2010 - set_2015
diff_in_2015 = set_2015 - set_2010

print(f"Only in 2010: {list(diff_in_2010)}")
print(f"Only in 2015: {list(diff_in_2015)}")

Only in 2010: []
Only in 2015: ['western islands', 'pioneer', 'marina south', 'lim chu kang', 'sungei kadut', 'paya lebar', 'north-eastern islands', 'southern islands', 'changi bay', 'seletar', 'boon lay', 'straits view', 'tuas', 'western water catchment', 'simpang', 'museum', 'tengah', 'central water catchment', 'marina east', 'orchard']


In [6]:
# get the planning areas that can be plotted together
population_2010_to_plot = population_pln_area_2010.copy()
population_2010_to_plot = population_2010_to_plot[["planning_area", "total", "total_above_60"]]
print(population_2010_to_plot.shape)

population_2015_to_plot = population_pln_area_2015[
    ~population_pln_area_2015["planning_area"].isin(diff_in_2015)
].copy()
population_2015_to_plot = population_2015_to_plot[["planning_area", "total", "total_above_60"]]
print(population_2015_to_plot.shape)

population_2020_to_plot = population_pln_area_2020[
    ~population_pln_area_2020["planning_area"].isin(diff_in_2015)
].copy()
population_2020_to_plot = population_2020_to_plot[["planning_area", "total", "total_above_60"]]
print(population_2020_to_plot.shape)

(35, 3)
(35, 3)
(35, 3)


In [7]:
def plot_graph_to_png(combined_remaining_df, planning_areas_to_plot, output_folder,
                      area_col_name, population_data_type, ylabel_name,
                      output_file_name):
    # 1.1. DETERMINE GRID SIZE
    num_plots = len(planning_areas_to_plot)
    # Calculate the number of columns and rows for a square-like layout
    cols = int(np.ceil(np.sqrt(num_plots)))
    rows = int(np.ceil(num_plots / cols))

    # 1.2. CREATE A SINGLE FIGURE WITH SUBPLOTS
    # Scale the figure size based on the number of subplots
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    # Flatten the axes array for easier iteration
    axes = axes.flatten()

    # plot for total population
    # 1.3. LOOP AND PLOT ONTO SUBPLOTS
    for i, area in enumerate(planning_areas_to_plot):
        # Filter the combined DataFrame for the current area
        area_data = combined_remaining_df[combined_remaining_df[area_col_name] == area]
        
        # Select the correct subplot axes using the index 'i'
        ax = axes[i]
        
        # Plot the population change
        ax.plot(area_data['year'], area_data[population_data_type], 
                marker='o', linestyle='-', color='indianred', linewidth=2)
        
        # Customize the subplot
        ax.set_title(area, fontsize=14) 
        ax.set_xlabel("Year", fontsize=12)
        ax.set_ylabel(ylabel_name, fontsize=12)
        
        # Set x-ticks to only show the specific years
        ax.set_xticks(area_data['year'])
        ax.ticklabel_format(style='plain', axis='y') # Avoid scientific notation
        ax.grid(True, linestyle='--', alpha=0.6)
        
        ax.ticklabel_format(style='plain', axis='y')
        ax.grid(True, linestyle='--', alpha=0.6)

    # 1.4. HIDE UNUSED SUBPLOTS 
    for j in range(num_plots, len(axes)):
        fig.delaxes(axes[j])

    # 1.5. ADJUST LAYOUT AND SAVE THE SINGLE FILE
    # Add a title for the entire figure
    plt.suptitle(f"{ylabel_name} Change for Planning Areas", fontsize=16, y=1.02)
    plt.tight_layout()

    # Define the final single filename
    final_filename = output_file_name
    filepath = os.path.join(output_folder, final_filename)
        
    plt.savefig(filepath)
    plt.close(fig)

In [9]:
# Add a 'year' column to each DataFrame
population_2010_to_plot['year'] = 2010
population_2015_to_plot['year'] = 2015
population_2020_to_plot['year'] = 2020

# Concatenate them into a single long-format DataFrame
combined_df = pd.concat([
    population_2010_to_plot, 
    population_2015_to_plot, 
    population_2020_to_plot
], ignore_index=True)

# Sort by planning area and year
combined_df = combined_df.sort_values(by=['planning_area', 'year']).reset_index(drop=True)

output_folder = "population_area_changes"
# Create the directory if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

planning_areas_to_plot = combined_df['planning_area'].unique()

# plot for general population
plot_graph_to_png(combined_df, planning_areas_to_plot, output_folder,
                  "planning_area", "total", "Total Population",
                  "all_pln_area_population_2010_2020.png")

# plot for population above 60
plot_graph_to_png(combined_df, planning_areas_to_plot, output_folder,
                  "planning_area", "total_above_60", "Population above 60",
                  "all_pln_area_above_60_2015_2020.png")

#### get the remaining planning areas of 2015 and 2020 that are not present in 2010 which will be plotted together

In [10]:
# get the remaining planning areas of 2015 and 2020 that are not present in 2010 which will be plotted together
remaining_population_2015_to_plot = population_pln_area_2015[
    population_pln_area_2015["planning_area"].isin(diff_in_2015)
].copy()
remaining_population_2015_to_plot = remaining_population_2015_to_plot[["planning_area", "total", "total_above_60"]]
print(remaining_population_2015_to_plot.shape)

remaining_population_2020_to_plot = population_pln_area_2020[
    population_pln_area_2020["planning_area"].isin(diff_in_2015)
].copy()
remaining_population_2020_to_plot = remaining_population_2020_to_plot[["planning_area", "total", "total_above_60"]]
print(remaining_population_2020_to_plot.shape)

(20, 3)
(20, 3)


In [12]:
# Add a 'year' column to each DataFrame
remaining_population_2015_to_plot['year'] = 2015
remaining_population_2020_to_plot['year'] = 2020

# Concatenate them into a single long-format DataFrame
combined_remaining_df = pd.concat([
    remaining_population_2015_to_plot, 
    remaining_population_2020_to_plot
], ignore_index=True)


# --- 3. CREATE OUTPUT FOLDER ---

output_folder = "population_area_changes"
os.makedirs(output_folder, exist_ok=True)


# --- 4. LOOP, PLOT, AND SAVE ---

planning_areas_to_plot = combined_remaining_df['planning_area'].unique()

# plot for general population
plot_graph_to_png(combined_remaining_df, planning_areas_to_plot, output_folder,
                  "planning_area", "total", "Total Population",
                  "remaining_pln_area_population_2015_2020.png")

# plot for population above 60
plot_graph_to_png(combined_remaining_df, planning_areas_to_plot, output_folder,
                  "planning_area", "total_above_60", "Population above 60",
                  "remaining_pln_area_above_60_2015_2020.png")


#### Population and above 60 by subzone

In [13]:
population_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/visualisation_data/subzone_with_population_data_2010.gpkg")
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/visualisation_data/subzone_with_population_data_2015.gpkg")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/visualisation_data/subzone_with_population_data_2020.gpkg")

population_subzone_2010  = gpd.read_file(population_2010_filepath)
population_subzone_2015  = gpd.read_file(population_2015_filepath)
population_subzone_2015 = population_subzone_2015.rename(columns = {"total_total": "total"})
population_subzone_2020  = gpd.read_file(population_2020_filepath)

In [14]:
subzone_2010 = population_subzone_2010["subzone"].tolist()
print(f"Number of planning areas in 2010 data: {len(subzone_2010)}")
subzone_2015 = population_subzone_2015["subzone"].tolist()
print(f"Number of planning areas in 2015 data: {len(subzone_2015)}")
subzone_2020 = population_subzone_2020["subzone"].tolist()
print(f"Number of planning areas in 2020 data: {len(subzone_2020)}")

if subzone_2015.sort() == subzone_2020.sort():
    print("The subzones for resident population data in 2015 and 2020 are the same")
if subzone_2020.sort() == subzone_2015.sort():
    print("The subzones for resident population data in 2020 and 2015 are the same")
else:
    print("NOT THE SAME")

Number of planning areas in 2010 data: 191
Number of planning areas in 2015 data: 323
Number of planning areas in 2020 data: 332
The subzones for resident population data in 2015 and 2020 are the same
The subzones for resident population data in 2020 and 2015 are the same


In [15]:
set_2010 = set(subzone_2010)
set_2015 = set(subzone_2015)
set_2020 = set(subzone_2020)

# diff_in_2010v2015 = set_2010 - set_2015
# diff_in_2010v2020 = set_2010 - set_2020
# diff_in_2015v2010 = set_2015 - set_2010
# diff_in_2015v2020 = set_2015 - set_2020
# diff_in_2020v2015 = set_2020 - set_2015
# diff_in_2020v2010 = set_2020 - set_2010

# print(f"Only in 2010: {list(diff_in_2010v2015)}")
# print(f"Only in 2010: {list(diff_in_2010v2020)}")
# print(f"Only in 2015: {list(diff_in_2015v2010)}")
# print(f"Only in 2015: {list(diff_in_2015v2020)}")
# print(f"Only in 2020: {list(diff_in_2020v2015)}")
# print(f"Only in 2020: {list(diff_in_2020v2010)}")

# get the common subzones between the 3 datasets
common_subzones = set_2010.intersection(set_2015, set_2020)
print(len(common_subzones))

163
